configuration

In [1]:
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm import tqdm

import torch
import torchvision.transforms as transforms

import timm

# =========================================================
# CONFIGURATION
# =========================================================

RAW_DIR = Path("../data/resized")

VALID_EXTENSIONS = [
    ".png",
    ".jpg",
    ".jpeg",
    ".tif",
    ".bmp"
]

BATCH_SIZE = 32

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", DEVICE)

c:\Users\rohit rawat\Desktop\iitr_internship\image_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


load the model

In [2]:
# =========================================================
# LOAD EFFICIENTNET-B0
# =========================================================

# pretrained model
model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=0  # remove classifier head
)

model.eval()

model.to(DEVICE)

EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bi

some transformation 

In [3]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

collect the images

In [4]:
# =========================================================
# COLLECT IMAGE PATHS
# =========================================================

image_paths = [

    p for p in RAW_DIR.glob("**/*")

    if p.suffix.lower() in VALID_EXTENSIONS
]

print(f"Total images: {len(image_paths)}")

Total images: 47770


feature extraction

In [5]:
# =========================================================
# FEATURE EXTRACTION
# =========================================================

features = []

paths = []

# disable gradients
with torch.no_grad():

    for path in tqdm(image_paths):

        try:

            # open image
            img = Image.open(path).convert("RGB")

            # transform image
            img_tensor = transform(img)

            # add batch dimension
            img_tensor = img_tensor.unsqueeze(0)

            # move to GPU/CPU
            img_tensor = img_tensor.to(DEVICE)

            # extract features
            feature_vector = model(img_tensor)

            # move back to cpu
            feature_vector = (
                feature_vector
                .cpu()
                .numpy()
                .flatten()
            )

            # store
            features.append(feature_vector)

            paths.append(str(path))

        except Exception as e:

            print(f"Error with {path}: {e}")

# =========================================================
# CONVERT TO NUMPY
# =========================================================

features = np.array(features)

print("\nFeature matrix shape:")
print(features.shape)

100%|██████████| 47770/47770 [26:25<00:00, 30.12it/s]



Feature matrix shape:
(47770, 1280)


In [6]:
# =========================================================
# SAVE FEATURES
# =========================================================

np.save(
    "../embeddings/efficientnet_b0_features.npy",
    features
)

np.save(
    "../embeddings/image_paths.npy",
    paths
)

print("\nFeatures saved successfully!")


Features saved successfully!
